# Qwen3.8-Flash-Next(176B MoE,6B active)· llama.cpp · G4

- 量化:Unsloth **UD-Q4_K_XL**(111GB,4 個分片)
- `--fit on`:llama.cpp 自動把放不下顯存的部分(主要是 N-gram 嵌入表)放到系統 RAM
- 參考實測(DataCamp,同款 RTX PRO 6000 96GB、131K context):約 80 tok/s,context 變長後約 64 tok/s
- 提供 OpenAI 相容 API(含工具呼叫),經 Cloudflare tunnel 給 omp 用

**執行階段:G4 GPU**。下載和編譯同時進行,開機到能用約 20–30 分鐘,這段時間照常扣單元。

## 1. 設定

In [ ]:
QUANT    = "UD-Q4_K_XL"
REPO     = "unsloth/Qwen3.8-Flash-Next-GGUF"
MODEL_DIR = "/content/Qwen3.8-Flash-Next-GGUF"
CTX      = 131072          # 參考設定;想更長可以試 262144,看 log 的記憶體
ALIAS    = "qwen3.8-flash-next"
PORT     = 8080

import secrets
API_KEY = secrets.token_urlsafe(24)
print("API_KEY =", API_KEY)

## 2. 檢查硬體

In [ ]:
import subprocess, shutil
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"]).decode().strip()
print("GPU :", gpu)
cc = gpu.split(",")[-1].strip().replace(".", "")
ram = int(open("/proc/meminfo").read().split()[1]) / 1024 / 1024
disk = shutil.disk_usage("/content").free / 1024**3
print(f"RAM : {ram:.0f} GB   Disk free: {disk:.0f} GB")
assert int(gpu.split(",")[1].split()[0]) > 90000, "需要 96GB 的 G4"
assert ram > 60, "系統 RAM 不夠放卸載的部分"
assert disk > 125, "磁碟不夠放 111GB 的 GGUF"

## 3. 背景下載模型(和下一格的編譯同時跑)

In [ ]:
!pip install -q -U "huggingface_hub[hf_xet]"
import os
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"   # 不留 xet 快取,省磁碟
dl_log = open("/content/download.log", "w")
dl = subprocess.Popen(
    ["hf", "download", REPO, "--include", f"{QUANT}/*", "--local-dir", MODEL_DIR],
    stdout=dl_log, stderr=subprocess.STDOUT, env=os.environ.copy())
print("下載中,PID", dl.pid)

## 4. 編譯 llama.cpp(CUDA,只編這張卡的架構,約 5–10 分鐘)

In [ ]:
!apt-get -qq install -y cmake > /dev/null
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={cc} -DLLAMA_CURL=OFF > /content/cmake.log 2>&1 && echo configured
!cmake --build /content/llama.cpp/build --config Release -j$(nproc) --target llama-server > /content/build.log 2>&1 && echo built || tail -n 30 /content/build.log
!ls -la /content/llama.cpp/build/bin/llama-server

## 5. 等下載完成

In [ ]:
import time, glob
while dl.poll() is None:
    got = sum(os.path.getsize(f) for f in glob.glob(f"{MODEL_DIR}/**/*", recursive=True) if os.path.isfile(f)) / 1024**3
    print(f"\r已下載 {got:.1f} GB / 約 111 GB", end="")
    time.sleep(15)
print("\n下載結束,exit code", dl.returncode)
shards = sorted(glob.glob(f"{MODEL_DIR}/{QUANT}/*.gguf"))
print(shards)
assert dl.returncode == 0 and shards, "下載失敗,看 /content/download.log"

## 6. 啟動 llama-server

In [ ]:
import urllib.request
cmd = [
    "/content/llama.cpp/build/bin/llama-server",
    "-m", shards[0],                 # 給第一個分片,其餘會自動載入
    "--alias", ALIAS,
    "--host", "127.0.0.1", "--port", str(PORT),
    "--api-key", API_KEY,
    "--ctx-size", str(CTX),
    "--parallel", "1",
    "--flash-attn", "on",
    "--fit", "on", "--fit-target", "4096",
    "--jinja",                       # 工具呼叫需要
    "--batch-size", "1024", "--ubatch-size", "512",
    "--temp", "1.0", "--top-p", "0.95", "--top-k", "20", "--min-p", "0.0",
]
srv_log = open("/content/llama.log", "w")
srv = subprocess.Popen(cmd, stdout=srv_log, stderr=subprocess.STDOUT)

def ready():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3).status == 200
    except Exception:
        return False

t0 = time.time()
while not ready():
    if srv.poll() is not None:
        raise SystemExit("llama-server 掛了,執行下一格看 log")
    time.sleep(10); print(f"\r載入中… {int(time.time()-t0)}s", end="")
print(f"\n✅ ready ({int(time.time()-t0)}s)")

In [ ]:
# 看模型怎麼分配到 GPU / CPU,以及錯誤訊息
!grep -iE "offload|CUDA0|CPU_Mapped|model buffer|KV|fit|error|failed" /content/llama.log | tail -n 40

## 7. 測試

In [ ]:
!pip install -q openai
from openai import OpenAI
client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key=API_KEY)
t0 = time.time()
r = client.chat.completions.create(model=ALIAS, max_tokens=512,
    messages=[{"role": "user", "content": "用 Python 寫一個 LRU cache,附簡短說明"}])
dt = time.time() - t0
print(r.choices[0].message.content[:1500])
print(f"\n{r.usage.completion_tokens} tokens / {dt:.1f}s ≈ {r.usage.completion_tokens/dt:.1f} tok/s(含首字延遲)")

## 8. 對外開放給 omp(Cloudflare tunnel)

In [ ]:
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
import re
tlog = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(2)
print(f'''貼到 ~/.omp/agent/models.yml(Windows:%USERPROFILE%\\.omp\\agent\\models.yml):

providers:
  colab-next:
    baseUrl: {url}/v1
    api: openai-completions
    apiKey: "{API_KEY}"
    models:
      - id: {ALIAS}
        name: Qwen3.8 Flash-Next (Colab)
        contextWindow: {CTX}
        maxTokens: 32768
        reasoning: true
''')

## 9. 看每次請求的速度

In [ ]:
# prompt eval = 讀入速度,eval = 生成速度(tok/s)
!grep -E "prompt eval time|       eval time" /content/llama.log | tail -n 10

## 10.(選用)把 GGUF 存到 Drive,下次不用重下

111GB 複製到 Drive 要一段時間,而且下次從 Drive 複製回來不一定比從 Hugging Face 下載快。建議先比較這次的下載時間再決定。

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# !rsync -a --info=progress2 {MODEL_DIR}/ /content/drive/MyDrive/models/Qwen3.8-Flash-Next-GGUF/

## 11. 收工

In [ ]:
for p in ("tunnel", "srv"):
    if p in globals(): globals()[p].terminate()
from google.colab import runtime
runtime.unassign()